# Case de tráfego

**Oficina Visão Computacional Aplicada à Cidade**
Semana do Urbanismo de Canaã dos Carajás, 11/11/2026

Vamos fazer uma câmera de trânsito virar dados de cidade. Num cruzamento de Bellevue (EUA), o computador vai **detectar** os veículos, **rastrear** cada um ao longo de 40 segundos e contar **de onde veio e para onde foi**. O resultado é a **Matriz de movimentos**, a mesma tabela que um engenheiro de tráfego usa para regular um semáforo.

Rode as células **na ordem, de cima para baixo**, acompanhando o projetor. Para rodar uma célula, clique no ▶ à esquerda dela (ou aperte Shift + Enter).

| Etapa | O que acontece |
|---|---|
| 0 | Preparar o ambiente |
| 1 | A imagem é uma tabela de números |
| 2 | Detectar os veículos na Cena-âncora |
| 3 | Mexer no limiar de confiança |
| 4 | Desenhar as Zonas por Aproximação |
| 5 | Rastrear e contar os 40 segundos |
| 6 | Pular: usar o Resultado pronto |
| 7 | Assistir e ler o vídeo de saída |

# 0. Preparar o ambiente

Baixa o código da Oficina, instala os pacotes e baixa o vídeo, a Cena-âncora e o modelo. Leva cerca de 1 minuto e termina com uma checklist ✅.

In [ ]:
#@title 0. Preparar o ambiente
import os, sys, subprocess
if "google.colab" in sys.modules:
    PASTA = "/content/oficina-vc-canaa"
    if not os.path.isdir(PASTA + "/.git"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "v2026.11.11",
                        "https://github.com/andrematte/oficina-vc-canaa", PASTA], check=True)
else:
    PASTA = os.path.abspath(".")  # Modo local: o notebook roda de dentro do repositório
if PASTA not in sys.path:
    sys.path.insert(0, PASTA)
from oficina_vc import preparar
preparar("case")

# 1. A imagem é uma tabela de números

Esta é a **Cena-âncora**, o quadro do cruzamento que acompanhou a teoria e o desafio "Que veículo é esse?". Para o computador, ela é uma tabela de números: uma linha para cada fileira de pixels, uma coluna para cada pixel e três números por pixel, um para cada cor (de 0 a 255).

O OpenCV guarda as cores na ordem **azul, verde, vermelho**. O pixel abaixo fica na lateral do ônibus: muito verde, algum azul e quase nada de vermelho.

In [ ]:
import cv2
from oficina_vc import caminho, mostrar

quadro = cv2.imread(str(caminho("cena_ancora_bellevue_623.png")))

print("Formato:", quadro.shape, " (altura, largura, cores)")
print("Pixel da linha 317, coluna 642:", quadro[317, 642], " (azul, verde, vermelho)")
print("Um pedaço de 5 × 5 pixels, só o verde:")
print(quadro[315:320, 640:645, 1])
mostrar(quadro)

# 2. Detectar os veículos na Cena-âncora

Agora um **modelo pré-treinado** procura veículos nessa tabela de números. Ele foi treinado em milhares de fotos com caixas desenhadas à mão (o conjunto COCO, com 80 classes) e devolve, para cada objeto que encontra, uma caixa, uma classe e uma **confiança** de 0 a 1.

- `classes=VEICULOS`: só as 4 classes de veículo que o modelo conhece. Repare nos nomes: não existe "pickup" nem "van";
- `conf=0.25`: só as caixas com confiança de pelo menos 0,25;
- `imgsz=960`: o tamanho em que o modelo enxerga a imagem (maior acha mais veículos pequenos, lá no fundo, e demora mais).

Compare com o desafio "Que veículo é esse?": o que o modelo diz da van e da pickup?

In [ ]:
from ultralytics import YOLO
from oficina_vc import desenhar_caixas

modelo = YOLO(str(caminho("yolo26n.pt")))
VEICULOS = [2, 3, 5, 7]
print("Classes de veículo do modelo:", {i: modelo.names[i] for i in VEICULOS})

deteccoes = modelo(quadro, classes=VEICULOS, conf=0.25, imgsz=960, verbose=False)
print(f"O modelo encontrou {len(deteccoes[0].boxes)} veículos.")
mostrar(desenhar_caixas(quadro, deteccoes))

# 3. Mexer no limiar de confiança

Arraste o controle e veja as caixas mudarem. A célula roda sozinha a cada mudança.

- Limiar **baixo**: o modelo acha mais veículos, mas começa a ver veículos onde não há (falsos positivos) ou duas caixas no mesmo carro.
- Limiar **alto**: sobram só as caixas certeiras, e os carros pequenos do fundo somem (falsos negativos).

Que valor parece justo para contar este cruzamento?

In [ ]:
#@title 3. Limiar de confiança { run: "auto" }
CONFIANCA = 0.25 #@param {type:"slider", min:0.05, max:0.95, step:0.05}

deteccoes = modelo(quadro, classes=VEICULOS, conf=CONFIANCA, imgsz=960, verbose=False)
print(f"Limiar {CONFIANCA:.2f}: {len(deteccoes[0].boxes)} veículos.")
mostrar(desenhar_caixas(quadro, deteccoes))

# 4. Desenhar as Zonas por Aproximação

Para virar indicador, a detecção precisa de um lugar no espaço. O cruzamento tem quatro **Aproximações**, os braços por onde os veículos chegam e saem: Norte (em cima), Leste (à direita), Sul (embaixo) e Oeste (à esquerda). Cada uma ganha uma **Zona**, um polígono desenhado sobre a pista.

A regra da contagem:

- a Zona onde o veículo aparece primeiro é a **origem**;
- a próxima Zona em que ele entra é o **destino**;
- o par origem → destino é o **Movimento** dele: **Reto**, **Conversão à direita**, **Conversão à esquerda** ou **Retorno**. Quem vem do Norte anda rumo ao sul: segue reto para o Sul, tem o Oeste à direita e o Leste à esquerda.

As posições são pixels `(x, y)`: `x` cresce para a direita e `y` cresce **para baixo**, a partir do canto de cima à esquerda. A conta é feita pelo ponto do meio da base de cada caixa, os "pés" do veículo.

In [ ]:
from oficina_vc import criar_zonas, desenhar_zonas

ZONAS = {                                                           # vértices de cada Zona
    "Norte": [(495, 55), (725, 55), (805, 235), (470, 235)],
    "Leste": [(890, 215), (1275, 225), (1275, 410), (1120, 420), (990, 330)],
    "Sul":   [(540, 625), (1130, 560), (1200, 695), (560, 695)],
    "Oeste": [(0, 290), (300, 278), (445, 252), (455, 345), (310, 360), (310, 575), (0, 580)],
}

zonas = criar_zonas(ZONAS)
mostrar(desenhar_zonas(quadro, zonas))

Para casa: troque os números de `ZONAS`, rode esta célula para ver onde ficaram e rode a etapa 5 de novo. Os nomes das Zonas precisam continuar Norte, Leste, Sul e Oeste: é deles que sai o nome de cada Movimento.

# 5. Rastrear e contar os 40 segundos

Agora o vídeo inteiro: 40 segundos de câmera fixa. Para cada quadro, o computador:

1. **detecta** os veículos, como na etapa 2;
2. **rastreia**: liga cada caixa ao mesmo veículo do quadro anterior e dá a ele um número, o **ID**;
3. **conta**: anota a Zona de origem de cada ID e, quando ele chega a outra Zona, soma um Movimento na **Matriz de movimentos**;
4. desenha tudo e grava o **vídeo de saída**, com o painel de Métricas de vídeo ao lado.

Algumas escolhas para caber no tempo da aula:

- `passo=2`: analisa 1 quadro a cada 2 (metade do trabalho, a mesma Matriz);
- `confianca=0.05`: o rastreador (ByteTrack) aceita caixas fracas, mas só para continuar o rastro de um veículo que já foi visto com confiança. É isso que segura o ID do carro meio escondido atrás do ônibus ou na curva.

Leva alguns minutos (a barra mostra quanto falta). **Se a barra estiver lenta demais, não espere**: clique no ■ ao lado desta célula para interromper e rode a etapa 6.

In [ ]:
from oficina_vc import criar_rastreador, processar_video

rastreador = criar_rastreador()

resultado = processar_video(
    caminho("bellevue-ne8th_2762-2802s.mp4"),
    modelo,
    rastreador,
    zonas,
    confianca=0.05,
    classes=VEICULOS,
    imgsz=960,
    passo=2,
)
resultado.resumo()

# 6. Pular: usar o Resultado pronto

**Só se a etapa 5 não terminou.** O Resultado pronto é o mesmo processamento, feito antes da Oficina, com os mesmos números. Rode esta célula e siga para a etapa 7.

In [ ]:
#@title 6. Pular: usar o Resultado pronto
from oficina_vc import resultado_pronto

resultado = resultado_pronto()
resultado.resumo()

# 7. Assistir e ler o vídeo de saída

À esquerda, o cruzamento com as caixas e os **Rastros** (o caminho dos últimos 2 segundos de cada veículo). Cada etiqueta mostra o ID e o Movimento: `#12 N→` já tem origem no Norte; `#12 N→L` foi contado como Norte → Leste. À direita, o painel de **Métricas de vídeo**, com a Matriz de movimentos enchendo ao vivo.

In [ ]:
from oficina_vc import mostrar_video

mostrar_video(resultado)

Leia o vídeo com o painel:

1. **Matriz de movimentos**: de qual Aproximação vem a maior parte dos veículos? Quantos seguem reto e quantos convertem?
2. **Do número à decisão**: o que essa Matriz justifica no semáforo? Pense no tempo de verde de cada Aproximação e na seta de conversão à esquerda.
3. **Ache o ônibus**: entre 17 s e 21 s, um ônibus verde e branco desce do Norte para o Sul. Ele entrou na Matriz como ônibus?
4. **Troca de ID**: olhe a esquina de cima à esquerda, onde quem vem do Norte converte para o Oeste. Algum veículo troca de número na curva? O que acontece com o Movimento dele na Matriz?
5. **Parados no semáforo**: as filas do Oeste e do Leste ficam paradas quase o vídeo todo. Elas entram na Matriz? Por quê?

**E se quisermos saber _que tipo_ de veículo passa?** A Matriz diz que quase tudo é "carro": a van de serviço com escada no teto (3 s a 8 s, Norte → Oeste), o caminhão-plataforma (5 s a 10 s, Norte → Leste) e as pickups entram como carro, porque o modelo só conhece as classes com que foi treinado. Para uma contagem de tráfego, essa diferença pesa (pavimento, vagas de carga, emissões). É o que o próximo notebook ensina, com fine-tuning: separar Carro, Pickup, Van, Caminhão e Ônibus.

Para guardar suas mudanças: **Arquivo › Salvar uma cópia no Drive**.